In [ ]:
import os
import sys

from sklearn.preprocessing import PolynomialFeatures

sys.path.append("..")
from src import *

# Feature Extraction
##### ℹ️ This notebook outputs 4 files in the `dataset/1-preprocessed` folder
In this notebook, we're going to create features derived from the original 7.
- Combination of polynomials up to the third order
- $\dfrac{ng}{np}$ ratio
- Normalization of density altitude
- Normalization of air density

In [2]:
# 1s
train_x, train_y = read_dataset(stage='original')
valid_x = read_dataset(stage='original', dataset='valid')
test_x = read_dataset(stage='original', dataset='test')

In [3]:
# 0s
# Constants for ISA model
T0 = 288.15  # Sea-level standard temperature (K)
p0 = 101325  # Sea-level standard atmospheric pressure (Pa)
a = 0.0065  # Temperature lapse rate (K/m)
g = 9.80665  # Gravitational acceleration (m/s^2)
R = 8.3144598  # Universal gas constant (J/(mol·K))
RS = 287.05  # Specific gas constant for air (J/(kg·K))

In [4]:
# 1s
def extract_features(df) -> pd.DataFrame:
    new_df = df.copy()
    to_poly = df  # df[['oat', 'mgt', 'pa', 'np', 'ng']] <--- More interpretable
    poly = PolynomialFeatures(degree=3, include_bias=False)
    poly_features = poly.fit_transform(to_poly)
    poly_df = (pd.DataFrame(poly_features, columns=poly.get_feature_names_out(to_poly.columns))
               .drop(to_poly.columns, axis=1))
    new_df = pd.concat([new_df, poly_df], axis=1)

    # Add NP/NG ratio feature
    new_df['np_ng_ratio'] = df['np'] / df['ng']

    # Air density formula: da = 1.2376 * pa + 118.8 * oat - 1782
    new_df['da'] = (1.2376 * df['pa']) + (118.8 * (df['oat'] + 273.15)) - 1782

    # Compute pressure at altitude (h in meters, assuming pressure altitude in dataset is in feet)
    new_df['h_m'] = df['pa'] * 0.3048  # Convert pressure altitude from feet to meters
    new_df['P'] = p0 * (1 - (a * new_df['h_m']) / T0) ** (g / (R * a))

    # Compute air density (rho), ensuring valid values
    new_df['rho'] = new_df['P'] / (RS * (df['oat'] + 273.15))

    # Normalize original features based on air density (rho)
    for col in df.columns:
        new_df[f'{col}_norm_da'] = df[col] / new_df['da']
        new_df[f'{col}_air_density'] = df[col] / new_df['rho']
    return new_df


new_train_x = extract_features(train_x)
new_valid_x = extract_features(valid_x)
new_test_x = extract_features(test_x)
new_test_x.head()

,trq_measured,oat,mgt,pa,ias,np,ng,trq_measured^2,trq_measured oat,trq_measured mgt,...,mgt_norm_da,mgt_air_density,pa_norm_da,pa_air_density,ias_norm_da,ias_air_density,np_norm_da,np_air_density,ng_norm_da,ng_air_density
0,76.2,29.50,648.0,303.2760,0.0000,99.83,96.77,5806.44,2247.900,49377.60,...,0.018756,811.435169,0.008778,379.766686,0.000000,0.000000,0.002890,125.008600,0.002801,121.176823
1,63.3,18.75,595.4,464.8200,96.5625,100.01,93.61,4006.89,1186.875,37688.82,...,0.017789,880.122366,0.013887,687.098553,0.002885,142.739026,0.002988,147.835132,0.002797,138.374630
2,87.3,2.50,644.4,503.5296,119.4375,99.92,96.87,7621.29,218.250,56256.12,...,0.020400,944.186843,0.015940,737.780918,0.003781,175.002042,0.003163,146.404639,0.003067,141.935722
3,85.4,0.25,630.7,458.4192,121.2500,100.04,96.04,7293.16,21.350,53861.78,...,0.020173,866.248013,0.014662,629.625371,0.003878,166.533331,0.003200,137.402016,0.003072,131.908133
4,73.1,21.25,625.9,626.3640,111.4375,100.17,95.67,5343.61,1553.375,45753.29,...,0.018426,1142.359307,0.018440,1143.206175,0.003281,203.389783,0.002949,182.824943,0.002816,174.611783


## Discard $np \lt ng$
Test and Validation sets don't have any sample in this region. Except for 2 samples of the test set.

Why should the coming KAN bother with the $34.5\%$ of the train samples outside the testing area?

In [5]:
train_np_gt_ng_x = new_train_x[train_x['np'] >= train_x['ng']]
train_np_lt_ng_x = new_train_x[train_x['np'] < train_x['ng']]
train_np_gt_ng_y = train_y[train_x['np'] >= train_x['ng']]
train_np_lt_ng_y = train_y[train_x['np'] < train_x['ng']]

## Export to `.csv` files

In [6]:
# 1m 46s
os.makedirs('../dataset/1-preprocessed', exist_ok=True)
train_np_gt_ng_x.to_csv('../dataset/1-preprocessed/X_train.csv', index=False)
train_np_lt_ng_x.to_csv('../dataset/1-preprocessed/X_train_discarded.csv', index=False)
new_valid_x.to_csv('../dataset/1-preprocessed/X_valid.csv', index=False)
new_test_x.to_csv('../dataset/1-preprocessed/X_test.csv', index=False)
train_np_gt_ng_y.to_csv('../dataset/1-preprocessed/y.csv', index=False)
train_np_lt_ng_y.to_csv('../dataset/1-preprocessed/y_discarded.csv', index=False)